In [1]:
from transformers import AutoModelForMultipleChoice,AutoTokenizer,TrainingArguments,Trainer
import evaluate
from datasets import DatasetDict

In [2]:
from datasets import DatasetDict
ds = DatasetDict.load_from_disk("./C3/")
ds

DatasetDict({
    test: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer'],
        num_rows: 1625
    })
    train: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer'],
        num_rows: 11869
    })
    validation: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer'],
        num_rows: 3816
    })
})

In [3]:
ds.pop("test")

Dataset({
    features: ['id', 'context', 'question', 'choice', 'answer'],
    num_rows: 1625
})

In [4]:
model_name = "bert-base-chinese"
tokenizer = AutoTokenizer.from_pretrained(model_name,local_files_only = True)

In [15]:
ds['train']['choice'][2]

['会是昨天开的', '男的没有参加', '讨论得不热烈', '参加的人很少']

In [ ]:
def process_fn(dataset):
    context = []
    question_choice = []
    labels = []
    for idx in range(len(dataset['id'])):
        ctx = "\n".join(dataset['context'][idx])
        question = dataset['question'][idx]
        choices = dataset['choice'][idx]
        for choice in choices: 
            context.append(ctx)
            question_choice.append(question +" "+choice)
        if len(choices)<4:
            for _ in range(4-len(choices)):
                context.append(ctx)
                question_choice.append(question +" "+"无法确定")
        labels.append(choices.index(dataset["answe"][idx]))
    